In [6]:
import json
import math # Import math for handling potential division by zero
from scipy.optimize import linear_sum_assignment # For bipartite matching
import numpy as np # For cost matrix

def normalize_sense(sense_str):
    """
    Normalizes a sense string to the second level.
    E.g., "Contingency.Cause.Result" -> "Contingency.Cause"
    """
    parts = sense_str.split('.')
    if len(parts) > 2:
        return '.'.join(parts[:2])
    return sense_str

def get_covered_span_nos(start, end, all_span_nos):
    """
    Given a span (start, end) and a list of all span numbers in the document,
    returns the set of span numbers covered by the span [start, end].
    Handles float span numbers.
    """
    if start is None or end is None or not all_span_nos:
        return set() # Return empty set if span is missing or no span numbers available

    # Ensure all_span_nos is sorted for efficient checking
    sorted_span_nos = sorted(all_span_nos)

    covered = set()
    for span_no in sorted_span_nos:
        # Use a small tolerance for float comparison if needed, but direct comparison might be sufficient
        # depending on how span_no and start/end are generated.
        # Let's stick to direct comparison first.
        if start <= span_no <= end:
            covered.add(span_no)
        # Since all_span_nos is sorted, we can break early if span_no exceeds end
        elif span_no > end:
             break
    return covered

def calculate_partial_agreement(gold_sense, pred_sense, all_span_nos):
    """
    Calculates the partial agreement score between a gold and predicted sense
    based on their argument spans and the document's span numbers.
    Formula: (|I_Arg1| + |I_Arg2|) / (|U_all_Args|)
    """
    # Define empty sets for missing arguments to avoid errors
    gold_arg1_span_nos = set()
    gold_arg2_span_nos = set()
    pred_arg1_span_nos = set()
    pred_arg2_span_nos = set()

    # Get span numbers for gold arguments if they exist
    if 'Arg1_start' in gold_sense and 'Arg1_end' in gold_sense:
         gold_arg1_span_nos = get_covered_span_nos(gold_sense['Arg1_start'], gold_sense['Arg1_end'], all_span_nos)
    if 'Arg2_start' in gold_sense and 'Arg2_end' in gold_sense:
         gold_arg2_span_nos = get_covered_span_nos(gold_sense['Arg2_start'], gold_sense['Arg2_end'], all_span_nos)

    # Get span numbers for predicted arguments if they exist
    if 'Arg1_start' in pred_sense and 'Arg1_end' in pred_sense:
         pred_arg1_span_nos = get_covered_span_nos(pred_sense['Arg1_start'], pred_sense['Arg1_end'], all_span_nos)
    if 'Arg2_start' in pred_sense and 'Arg2_end' in pred_sense:
         pred_arg2_span_nos = get_covered_span_nos(pred_sense['Arg2_start'], pred_sense['Arg2_end'], all_span_nos)

    # Calculate intersections
    inter_arg1 = gold_arg1_span_nos.intersection(pred_arg1_span_nos)
    inter_arg2 = gold_arg2_span_nos.intersection(pred_arg2_span_nos)

    # Calculate union of all spans involved
    union_all_args = gold_arg1_span_nos.union(gold_arg2_span_nos).union(pred_arg1_span_nos).union(pred_arg2_span_nos)

    numerator = len(inter_arg1) + len(inter_arg2)
    denominator = len(union_all_args)

    if denominator == 0:
        # If there are no spans involved at all, agreement is undefined or 0.
        # Returning 0 seems reasonable in the context of evaluation.
        return 0.0
    else:
        return numerator / denominator

def load_data_and_spans(path, id_field="id", senses_field="Senses", spans_field="Spans"):
    """
    Load a JSONL file, extracting senses and span numbers.
    Normalizes gold senses.
    """
    senses_map = {}
    span_nos_map = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            key = obj.get(id_field)
            if key is None:
                 print(f"Warning: Missing ID field '{id_field}' in line: {line.strip()}")
                 continue # Skip lines without an ID

            # Load senses as a list of dicts
            senses_list = obj.get(senses_field, [])
            processed_senses = []
            for sense in senses_list:
                 if isinstance(sense, dict):
                      # Remove 'confidence' from predicted senses (harmless if not present)
                      sense.pop('confidence', None)
                      # Normalize sense label for gold data (harmless for pred data if already level 2)
                      sense['sense'] = normalize_sense(sense.get('sense', '')) # Normalize, handle missing sense key
                      processed_senses.append(sense)
                 else:
                      print(f"Warning: Unexpected sense format for key {key}: {sense}")
                      # Depending on expected data, might want to skip or handle differently
                      pass # Skip non-dictionary senses for now

            senses_map[key] = processed_senses

            # Load span numbers
            spans_list = obj.get(spans_field, [])
            span_nos = [span.get('span_no') for span in spans_list if isinstance(span, dict) and span.get('span_no') is not None]
            span_nos_map[key] = sorted(list(set(span_nos))) # Get unique sorted span numbers

    return senses_map, span_nos_map


def compute_per_item_scores(gold_senses_map, pred_senses_map, span_nos_map, partial_agreement_threshold=0.5):
    """
    For each key in the union of gold and pred maps,
    compute precision, recall, f1 using partial agreement and bipartite matching.
    """
    scores = {}
    all_keys = set(gold_senses_map) | set(pred_senses_map)

    for key in sorted(all_keys):
        gold_senses = gold_senses_map.get(key, [])
        pred_senses = pred_senses_map.get(key, [])
        all_span_nos = span_nos_map.get(key, [])

        num_gold = len(gold_senses)
        num_pred = len(pred_senses)

        tp = 0
        fp = 0
        fn = 0
        # Total_partial_agreement = 0 # If you wanted to sum partial scores instead of counting thresholded matches

        if num_gold == 0 and num_pred == 0:
            scores[key] = {
                "precision": 1.0, "recall": 1.0, "f1": 1.0,
                "tp": 0, "fp": 0, "fn": 0, "num_gold": 0, "num_pred": 0
            }
            continue # Perfect score if both are empty

        if num_gold > 0 and num_pred > 0:
            # Build a cost matrix for bipartite matching
            # We want maximum weight matching, so use negative partial agreement as cost
            # or subtract from a large value. Let's use negative partial agreement directly.
            cost_matrix = np.zeros((num_gold, num_pred))
            for i, gold_s in enumerate(gold_senses):
                for j, pred_s in enumerate(pred_senses):
                    # Only consider matching if normalized senses are the same
                    if gold_s.get('sense') == pred_s.get('sense'):
                         pa_score = calculate_partial_agreement(gold_s, pred_s, all_span_nos)
                         # Use negative score for min-cost matching
                         cost_matrix[i, j] = -pa_score
                    else:
                        # Assign a high cost (or very low negative weight) for sense mismatch
                        # This effectively prevents matching senses with different labels
                        cost_matrix[i, j] = 1.0 # Cost of 1.0 means partial agreement -1.0, lower than any valid score

            # Find the minimum cost matching using the Hungarian algorithm
            # row_ind, col_ind are the indices of matched gold and pred senses
            row_ind, col_ind = linear_sum_assignment(cost_matrix)

            # Count TP based on matched pairs meeting the partial agreement threshold
            matched_gold_indices = set()
            matched_pred_indices = set()

            for r, c in zip(row_ind, col_ind):
                # The score is the negative of the cost
                pa_score = -cost_matrix[r, c]
                # Check if the score is positive (meaning sense matched) and meets threshold
                if pa_score > 0 and pa_score >= partial_agreement_threshold:
                    tp += 1
                    # Total_partial_agreement += pa_score # If summing scores
                    matched_gold_indices.add(r)
                    matched_pred_indices.add(c)

            # FP are predicted senses that were not matched to a gold sense meeting criteria
            fp = num_pred - len(matched_pred_indices)

            # FN are gold senses that were not matched by any predicted sense meeting criteria
            fn = num_gold - len(matched_gold_indices)


        elif num_gold > 0 and num_pred == 0:
             # If gold has senses but pred has none, all gold senses are FN
             tp = 0
             fp = 0
             fn = num_gold
        elif num_gold == 0 and num_pred > 0:
             # If pred has senses but gold has none, all pred senses are FP
             tp = 0
             fp = num_pred
             fn = 0


        prec = tp / (tp + fp) if tp + fp > 0 else 0.0
        rec  = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

        scores[key] = {
            "precision": round(prec, 4),
            "recall":    round(rec, 4),
            "f1":        round(f1,  4),
            "tp": tp, "fp": fp, "fn": fn,
            "num_gold": num_gold, "num_pred": num_pred,
            # "total_partial_agreement": round(Total_partial_agreement, 4) # Optional: if you want to see sum of scores
        }
    return scores

if __name__ == "__main__":
    gold_path = "/content/processed_data_explicit.jsonl" # Assuming this contains "Doc" and "Senses" and "Spans"
    pred_path = "/content/(no_uplicate)output_merged_explicit_claude.jsonl" # Assuming this contains "id" and "Senses"

    # Load data, including span numbers from the gold file
    # Assuming gold file structure is {"Doc": ..., "Spans": [...], "Senses": [...]}
    gold_senses_map, gold_span_nos_map = load_data_and_spans(gold_path, id_field="Doc", senses_field="Senses", spans_field="Spans")

    # Load predicted senses (we don't need spans from the prediction file for this calculation)
    # Assuming pred file structure is {"id": ..., "Senses": [...], "confidence": ...}
    pred_senses_map, _ = load_data_and_spans(pred_path, id_field="id", senses_field="Senses", spans_field="Spans") # Pass spans_field, but the returned map is ignored

    # Ensure span numbers for all documents are available, using gold as the source
    # If a document exists in pred but not gold, it won't have span_nos.
    # We should ideally only evaluate on documents present in gold.
    # Let's filter pred_senses_map to only include keys in gold_senses_map
    # and use gold_span_nos_map for all span number lookups.
    pred_senses_map_filtered = {k: v for k, v in pred_senses_map.items() if k in gold_senses_map}


    # Compute scores using the gold span numbers map for all calculations
    # Set a partial agreement threshold (e.g., 0.5)
    scores = compute_per_item_scores(gold_senses_map, pred_senses_map_filtered, gold_span_nos_map, partial_agreement_threshold=0.5)


    # Print per-item scores
    for key, metrics in scores.items():
        # Added num_gold and num_pred for clarity
        print(f"{key}: precision={metrics['precision']}, "
              f"recall={metrics['recall']}, f1={metrics['f1']} "
              f"(TP={metrics['tp']}, FP={metrics['fp']}, FN={metrics['fn']}, "
              f"Gold={metrics['num_gold']}, Pred={metrics['num_pred']})")


    # Optionally save all scores to JSON
    with open("per_item_f1_scores_partial_agreement.json", "w", encoding="utf-8") as out:
        json.dump(scores, out, indent=2)

Furnace_troubleshooting: precision=0.061, recall=0.2453, f1=0.0977 (TP=13, FP=200, FN=40, Gold=53, Pred=213)
Garage_floor_crack_repair: precision=0.0522, recall=0.1892, f1=0.0819 (TP=7, FP=127, FN=30, Gold=37, Pred=134)
How_to_dig_basement: precision=0.0843, recall=0.2188, f1=0.1217 (TP=7, FP=76, FN=25, Gold=32, Pred=83)
Occupancy_sensor_part1: precision=0.0, recall=0.0, f1=0.0 (TP=0, FP=40, FN=8, Gold=8, Pred=40)
Occupancy_sensor_part2: precision=0.0, recall=0.0, f1=0.0 (TP=0, FP=84, FN=5, Gold=5, Pred=84)
PLC_output: precision=0.0127, recall=0.125, f1=0.023 (TP=1, FP=78, FN=7, Gold=8, Pred=79)
Pouring_concrete: precision=0.0145, recall=0.0714, f1=0.0241 (TP=2, FP=136, FN=26, Gold=28, Pred=138)
RV_Plumbing: precision=0.1029, recall=0.4118, f1=0.1647 (TP=7, FP=61, FN=10, Gold=17, Pred=68)
Simple_boiler_maintainence: precision=0.1765, recall=0.4286, f1=0.25 (TP=6, FP=28, FN=8, Gold=14, Pred=34)
Voltage_drop: precision=0.0738, recall=0.2474, f1=0.1137 (TP=24, FP=301, FN=73, Gold=97, Pred

In [11]:
# (Keep imports and helper functions: normalize_sense, get_covered_span_nos, calculate_partial_agreement, load_data_and_spans)

def compute_scores(gold_senses_map, pred_senses_map, span_nos_map, use_partial_agreement=True, partial_agreement_threshold=0.5):
    """
    Compute scores per item and overall, with an option for partial agreement.

    Args:
        gold_senses_map (dict): Map of doc_id to list of gold sense dicts.
        pred_senses_map (dict): Map of doc_id to list of predicted sense dicts.
        span_nos_map (dict): Map of doc_id to sorted list of unique span numbers.
        use_partial_agreement (bool): Whether to use partial agreement score (True)
                                     or exact match (False).
        partial_agreement_threshold (float): Threshold for partial agreement
                                             if exact match is False (not used
                                             if use_partial_agreement is True).

    Returns:
        dict: Contains 'per_item_scores' and 'overall_scores'.
    """
    per_item_scores = {}
    all_keys = sorted(list(set(gold_senses_map) | set(pred_senses_map)))

    # Accumulators for overall scores
    total_tp = 0.0 # Use float for partial agreement sums
    total_fp = 0.0
    total_fn = 0.0
    total_gold_count = 0 # Total number of gold senses across all items
    total_pred_count = 0 # Total number of predicted senses across all items

    for key in all_keys:
        gold_senses = gold_senses_map.get(key, [])
        pred_senses = pred_senses_map.get(key, [])
        all_span_nos = span_nos_map.get(key, []) # Span numbers from gold data

        num_gold = len(gold_senses)
        num_pred = len(pred_senses)

        # Initialize per-item counts/sums
        item_tp = 0.0
        item_fp = 0.0
        item_fn = 0.0

        total_gold_count += num_gold
        total_pred_count += num_pred

        if num_gold == 0 and num_pred == 0:
            # Perfect score if both are empty
            per_item_scores[key] = {
                "precision": 1.0, "recall": 1.0, "f1": 1.0,
                "tp": 0.0, "fp": 0.0, "fn": 0.0, "num_gold": 0, "num_pred": 0
            }
            # Overall counts remain unchanged (add 0 to totals)
            continue

        if use_partial_agreement:
            # --- Partial Agreement Logic ---
            if num_gold > 0 and num_pred > 0:
                # Build cost matrix: rows=gold, cols=pred
                # Cost is negative partial agreement for matching senses, high cost otherwise
                cost_matrix = np.full((num_gold, num_pred), 1.0) # Default high cost (e.g., 1.0, as PA is 0-1)
                for i, gold_s in enumerate(gold_senses):
                    for j, pred_s in enumerate(pred_senses):
                        # Assumes senses are already normalized during loading
                        if gold_s.get('sense') == pred_s.get('sense'):
                            pa_score = calculate_partial_agreement(gold_s, pred_s, all_span_nos)
                            cost_matrix[i, j] = -pa_score # Min cost matching wants minimum value

                # Find the minimum cost matching (maximum total partial agreement)
                row_ind, col_ind = linear_sum_assignment(cost_matrix)

                # Sum the partial agreement scores for the matched pairs
                item_partial_tp_sum = 0.0
                matched_gold_indices = set()
                matched_pred_indices = set()

                for r, c in zip(row_ind, col_ind):
                     pa_score = -cost_matrix[r, c] # Get the actual partial agreement score back
                     # Only consider matches where sense label was identical (pa_score > 0 from calculate_partial_agreement if spans overlap)
                     # and where it wasn't the default high cost (which represents sense mismatch)
                     if gold_senses[r].get('sense') == pred_senses[c].get('sense'):
                          item_partial_tp_sum += pa_score
                          matched_gold_indices.add(r) # Track indices to calculate FP/FN counts later if needed, though not strictly necessary for float TP/FP/FN
                          matched_pred_indices.add(c)


                # TP is the sum of partial agreement scores
                item_tp = item_partial_tp_sum
                # FP = Total Predicted - Sum of TP
                item_fp = num_pred - item_tp
                # FN = Total Gold - Sum of TP
                item_fn = num_gold - item_tp # Note: item_tp here is the sum of scores, not count

            elif num_gold > 0 and num_pred == 0:
                 # If gold has senses but pred has none, all gold senses are FN (score 1 each)
                 item_tp = 0.0
                 item_fp = 0.0
                 item_fn = num_gold # Count as FN
            elif num_gold == 0 and num_pred > 0:
                 # If pred has senses but gold has none, all pred senses are FP (score 1 each)
                 item_tp = 0.0
                 item_fp = num_pred # Count as FP
                 item_fn = 0.0

        else:
            # --- Exact Match Logic ---
            gold_sense_set = set()
            for gold_s in gold_senses:
                # Need a hashable representation. Use frozenset of items excluding Arg/explicit/sense
                # No, the *entire* sense dict (after removing confidence and normalizing sense)
                # defines the identity for exact match.
                # Let's ensure normalization happened during load_data_and_spans
                hashable_gold_s = frozenset(gold_s.items())
                gold_sense_set.add(hashable_gold_s)

            pred_sense_set = set()
            for pred_s in pred_senses:
                 # Ensure normalization happened during load_data_and_spans and confidence removed
                 hashable_pred_s = frozenset(pred_s.items())
                 pred_sense_set.add(hashable_pred_s)


            # Calculate integer counts for exact match
            item_tp = len(gold_sense_set.intersection(pred_sense_set))
            item_fp = len(pred_sense_set - gold_sense_set)
            item_fn = len(gold_sense_set - pred_sense_set)

        # Calculate per-item precision, recall, f1 (handles both float and int counts)
        item_prec = item_tp / (item_tp + item_fp) if (item_tp + item_fp) > 0 else 0.0
        item_rec = item_tp / (item_tp + item_fn) if (item_tp + item_fn) > 0 else 0.0
        item_f1 = (2 * item_prec * item_rec / (item_prec + item_rec)) if (item_prec + item_rec) > 0 else 0.0

        per_item_scores[key] = {
            "precision": round(item_prec, 4),
            "recall":    round(item_rec, 4),
            "f1":        round(item_f1,  4),
            "tp": round(item_tp, 4), # Round float TP for display
            "fp": round(item_fp, 4), # Round float FP for display
            "fn": round(item_fn, 4), # Round float FN for display
            "num_gold": num_gold,
            "num_pred": num_pred,
        }

        # Accumulate for overall scores
        total_tp += item_tp
        total_fp += item_fp
        total_fn += item_fn


    # --- Calculate Overall Scores ---
    overall_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    overall_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    overall_f1 = (2 * overall_prec * overall_rec / (overall_prec + overall_rec)) if (overall_prec + overall_rec) > 0 else 0.0

    overall_scores = {
        "precision": round(overall_prec, 4),
        "recall": round(overall_rec, 4),
        "f1": round(overall_f1, 4),
        "total_tp": round(total_tp, 4),
        "total_fp": round(total_fp, 4),
        "total_fn": round(total_fn, 4),
        "total_gold": total_gold_count,
        "total_pred": total_pred_count,
    }

    return {"per_item_scores": per_item_scores, "overall_scores": overall_scores}


# --- Main execution block ---
if __name__ == "__main__":
    gold_path = "/content/processed_data_explicit.jsonl" # Assuming this contains "Doc", "Spans", "Senses"
    pred_path = "/content/(no_uplicate)output_merged_explicit.jsonl" # Assuming this contains "id", "Senses"

    # Load data, including span numbers from the gold file
    gold_senses_map, gold_span_nos_map = load_data_and_spans(gold_path, id_field="Doc", senses_field="Senses", spans_field="Spans")

    # Load predicted senses (we don't need spans from the prediction file for this calculation)
    pred_senses_map, _ = load_data_and_spans(pred_path, id_field="id", senses_field="Senses") # spans_field is optional/ignored here


    # Filter predicted senses map to only include document IDs present in the gold map
    # This ensures we only evaluate on the gold standard set of documents and have spans available.
    eval_doc_ids = sorted(list(gold_senses_map.keys()))
    pred_senses_map_filtered = {k: pred_senses_map.get(k, []) for k in eval_doc_ids}
    gold_span_nos_map_filtered = {k: gold_span_nos_map.get(k, []) for k in eval_doc_ids} # Also filter spans map

    # --- Compute scores with Partial Agreement ---
    print("--- Computing Scores with Partial Agreement ---")
    partial_agreement_results = compute_scores(
        {k: gold_senses_map[k] for k in eval_doc_ids}, # Use filtered gold senses map
        pred_senses_map_filtered,
        gold_span_nos_map_filtered,
        use_partial_agreement=True
    )

    # Print overall partial agreement scores
    print("\nOverall Scores (Partial Agreement):")
    overall_pa = partial_agreement_results['overall_scores']
    print(f"Precision: {overall_pa['precision']}, Recall: {overall_pa['recall']}, F1: {overall_pa['f1']}")
    print(f"Total TP: {overall_pa['total_tp']}, Total FP: {overall_pa['total_fp']}, Total FN: {overall_pa['total_fn']}")
    print(f"Total Gold Senses: {overall_pa['total_gold']}, Total Predicted Senses: {overall_pa['total_pred']}")

    # Optionally print per-item partial agreement scores
    print("\nPer-Item Scores (Partial Agreement):")
    for key, metrics in partial_agreement_results['per_item_scores'].items():
         print(f"{key}: precision={metrics['precision']}, recall={metrics['recall']}, f1={metrics['f1']} "
               f"(TP={metrics['tp']}, FP={metrics['fp']}, FN={metrics['fn']}, "
               f"Gold={metrics['num_gold']}, Pred={metrics['num_pred']})")


    # Optionally save partial agreement scores to JSON
    with open("partial_agreement_f1_scores.json", "w", encoding="utf-8") as out:
         json.dump(partial_agreement_results, out, indent=2)


    # --- Compute scores with Exact Match ---
    print("\n--- Computing Scores with Exact Match ---")
    exact_match_results = compute_scores(
         {k: gold_senses_map[k] for k in eval_doc_ids}, # Use filtered gold senses map
         pred_senses_map_filtered,
         gold_span_nos_map_filtered, # span_nos_map is not strictly needed for exact match, but passed anyway
         use_partial_agreement=False
    )

    # Print overall exact match scores
    print("\nOverall Scores (Exact Match):")
    overall_exact = exact_match_results['overall_scores']
    print(f"Precision: {overall_exact['precision']}, Recall: {overall_exact['recall']}, F1: {overall_exact['f1']}")
    print(f"Total TP: {overall_exact['total_tp']}, Total FP: {overall_exact['total_fp']}, Total FN: {overall_exact['total_fn']}")
    print(f"Total Gold Senses: {overall_exact['total_gold']}, Total Predicted Senses: {overall_exact['total_pred']}")

    # Optionally print per-item exact match scores
    print("\nPer-Item Scores (Exact Match):")
    for key, metrics in exact_match_results['per_item_scores'].items():
         print(f"{key}: precision={metrics['precision']}, recall={metrics['recall']}, f1={metrics['f1']} "
               f"(TP={metrics['tp']}, FP={metrics['fp']}, FN={metrics['fn']}, "
               f"Gold={metrics['num_gold']}, Pred={metrics['num_pred']})")

    # Optionally save exact match scores to JSON
    with open("exact_match_f1_scores.json", "w", encoding="utf-8") as out:
         json.dump(exact_match_results, out, indent=2)

--- Computing Scores with Partial Agreement ---

Overall Scores (Partial Agreement):
Precision: 0.3491, Recall: 0.5022, F1: 0.4119
Total TP: 445.4179, Total FP: 830.5821, Total FN: 441.5821
Total Gold Senses: 887, Total Predicted Senses: 1276

Per-Item Scores (Partial Agreement):
Furnace_troubleshooting: precision=0.3793, recall=0.5231, f1=0.4398 (TP=64.8667, FP=106.1333, FN=59.1333, Gold=124, Pred=171)
Garage_floor_crack_repair: precision=0.4682, recall=0.6292, f1=0.5369 (TP=60.4, FP=68.6, FN=35.6, Gold=96, Pred=129)
How_to_dig_basement: precision=0.385, recall=0.5578, f1=0.4556 (TP=27.3333, FP=43.6667, FN=21.6667, Gold=49, Pred=71)
Occupancy_sensor_part1: precision=0.3056, recall=0.3735, f1=0.3361 (TP=10.0833, FP=22.9167, FN=16.9167, Gold=27, Pred=33)
Occupancy_sensor_part2: precision=0.3776, recall=0.2607, f1=0.3085 (TP=21.9, FP=36.1, FN=62.1, Gold=84, Pred=58)
PLC_output: precision=0.3119, recall=0.4838, f1=0.3793 (TP=23.7056, FP=52.2944, FN=25.2944, Gold=49, Pred=76)
Pouring_concr